In [ ]:
!which python

In [2]:
%load_ext autoreload
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell

In [3]:
# basic packages
import os
import re
import sys
import datetime
from typing import List, Dict, Tuple, Optional, Any
from itertools import combinations
from pathlib import Path
import glob
#import yaml
import tqdm
import ast
import multiprocessing as mp
from itertools import combinations, product
import pytest

In [4]:
# data science
import numpy as np
import scipy as sc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import math

from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression


In [5]:
from bintools.align.align import read_phylip, read_fasta, read_ali, write_phylip, write_fasta, write_fasta_from_align
from bintools.utils.utils import get_yaml_config
from bintools.align.align import ali
from bintools.phylobayes.alphabet import amino_acids_upper

from Bio.Align import MultipleSeqAlignment

In [6]:
ROOT_dir = Path(os.path.abspath(os.path.join(Path("../")))).__str__()
if ROOT_dir not in sys.path:
    sys.path.append(ROOT_dir)

In [7]:
from cogent3.core.profile import MotifFreqsArray
from cogent3 import get_moltype
from cogent3.draw.logo import get_logo
from cogent3.util.union_dict import UnionDict

In [8]:
from scripts.utils import *

In [9]:
from scripts.compute_WAG import AA, WAG_RR, WAG_Stat
from scripts.compute_LG import LG_RR, LG_Stat

In [11]:
dict_of_WAG = {}
dict_of_LG = {}
k = 0
total_WAG = 0
total_LG = 0
for a in range(0,20):
    for b in range(a+1,20):
        if a == b:
            pass
        else:
            dict_of_WAG[AA[a]+"<>"+AA[b]] = WAG_RR[k]
            total_WAG += dict_of_WAG[AA[a]+"<>"+AA[b]]
            dict_of_LG[AA[a]+"<>"+AA[b]] = LG_RR[k]
            total_LG += dict_of_LG[AA[a]+"<>"+AA[b]]

            k+=1 


dict_of_WAG = {k: v / total_WAG for k, v in dict_of_WAG.items()}
dict_of_LG = {k: v / total_LG for k, v in dict_of_LG.items()}

assert math.isclose(1,sum(dict_of_WAG.values()))
assert math.isclose(1,sum(dict_of_LG.values()))

df_LG_rho = pd.DataFrame.from_dict(data=dict_of_LG,orient="index",columns=["rho"])
df_WAG_rho = pd.DataFrame.from_dict(data=dict_of_WAG,orient="index",columns=["rho"])




In [ ]:
MODELID = ["JCAA"]
GENEID = ["BGLOBIN"]


list_of_df = []
for modelID in MODELID:
    for geneID in GENEID: 
        for repID in ["A",]:
            df: pd.DataFrame = pd.read_csv(f"{ROOT_dir}/outputs/empirical/site_profile/stats/AA/{geneID}-{modelID}-{repID}_suffstatmap.tsv",sep="\t")
            # df_post = df.loc[df["type"] == "post"]
            # df_pred = df.loc[df["type"] == "pred"]
            list_of_df += [df]
df_concat  = pd.concat(list_of_df)       

list_of_df_centroids = []
dict_of_stats = {}
l = 0
for modelID in MODELID:
    for typeID in ["post", "pred"]:
        for geneID in GENEID:
            for repID in ["A",]:
                df_cur = df_concat.loc[(df_concat["type"]==typeID)&(df_concat["modelID"]==modelID)&(df_concat["geneID"]==geneID)&(df_concat["repID"]==repID)][amino_acids_upper[:-1]]
                scaler = StandardScaler()
                df_cur_scaled = scaler.fit_transform(df_cur)
                
                for k in [10,60]:
                    kmeans = KMeans(n_clusters=k, random_state=42)

                    # Fit the model
                    kmeans.fit(df_cur_scaled)
                    centroids = kmeans.cluster_centers_
                    cluster_labels = kmeans.labels_

                    dict_of_count = {}

                    for i in range(k):
                        dict_of_count[i] = np.sum(cluster_labels == i) / len(cluster_labels)

                    centroids_inverse_transformed = scaler.inverse_transform(centroids)
                    df_centroids_inverse_transformed = pd.DataFrame(data=centroids_inverse_transformed, columns=amino_acids_upper[:-1])
                    df_centroids_inverse_transformed["typeID"] = [typeID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["modelID"] = [modelID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["geneID"] = [geneID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["repID"] = [repID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["k"] = [k] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["weight"] = list(dict_of_count.values())

                    entropy_mappings = compute_entropy(df_centroids_inverse_transformed[amino_acids_upper[:-1]])

                    list_of_df_centroids += [df_centroids_inverse_transformed[["typeID","geneID", "modelID", "repID","k","weight"] + amino_acids_upper[:-1]]]
                    # if k == 10:
                    #     scaled_C10_profiles = scaler.transform(df_C10_profiles)
                    #     cluster_labels = kmeans.predict(scaled_C10_profiles)
                    #     cluster_labels_set = set(cluster_labels)
                    #     entropy_mixture = compute_entropy(df_C10_profiles)
                    
                    # elif k == 60:
                    #     scaled_C60_profiles = scaler.transform(df_C60_profiles)
                    #     cluster_labels = kmeans.predict(scaled_C60_profiles)
                    #     cluster_labels_set = set(cluster_labels)
                    #     entropy_mixture = compute_entropy(df_C60_profiles)
                    
                    dict_of_stats[l] = {
                        "typeID": typeID, # "post", "pred
                        "modelID": modelID,
                        "geneID": geneID,
                        "repID": repID,
                        "k": "Nclusters"+str(k),
                        "cluster_labels": cluster_labels,
                        "cluster_weights": list(dict_of_count.values()),
                        "entropy_mappings_mean": entropy_mappings.mean(),
                        

                    }
                    l+=1
df_centroids = pd.concat(list_of_df_centroids)

list_of_profiles_post = []
list_of_profiles_pred = []
for row in df_centroids.iterrows():
    typeID = row[1][0]
    profileID =row[1][4]
    if profileID == 10 and typeID == "post":
        dict_of_profile = {}
        sum_ = 0
        for i in range(6,26):
            aaID = row[1].index[i]
            aaPref = row[1][i]
            # print(aaPref)
            dict_of_profile.update({aaID : aaPref})
            sum_ += aaPref
        assert abs(sum_ - 1) < 0.01, f"Sum is {sum_}, but it should be 1."
        list_of_profiles_post += [dict_of_profile]

    if profileID == 10 and typeID == "pred":
        dict_of_profile = {}
        sum_ = 0
        for i in range(6,26):
            aaID = row[1].index[i]
            aaPref = row[1][i]
            # print(aaPref)
            dict_of_profile.update({aaID : aaPref})
            sum_ += aaPref
        assert abs(sum_ - 1) < 0.01, f"Sum is {sum_}, but it should be 1."
        list_of_profiles_pred += [dict_of_profile]

In [ ]:



# Create a figure
fig = plt.figure(figsize=(15, 9))

# Define a GridSpec layout (3 rows, 2 columns)
gs = gridspec.GridSpec(8, 2, width_ratios=[1, 1])

# Left section: 3 subplots stacked vertically in the first column
ax1 = plt.subplot(gs[0:2, 0])  # First subplot in the first column
ax2 = plt.subplot(gs[2:6, 0])  # Second subplot in the first column
ax3 = plt.subplot(gs[6:8, 0])  # Third subplot in the first column

# Right section: 2 subplots stacked vertically in the second column
ax4 = plt.subplot(gs[0:4, 1])  # First and second row, merged
ax5 = plt.subplot(gs[4:8, 1])    # Third row, second column

# Set titles or plot some data for clarity
# ax1.set_title('Left Plot 1')
# ax2.set_title('Left Plot 2')
# ax3.set_title('Left Plot 3')
# ax4.set_title('Right Plot 1 (Merged Rows)')
# ax5.set_title('Right Plot 2')

##############
# A
##############

df: pd.DataFrame = pd.read_csv(f"{ROOT_dir}/outputs/empirical/site_rates/stats/AA/{geneID}-{modelID}-A_suffstatmap.tsv",sep="\t")
df_post = df.loc[(df["modelID"] == modelID) & (df["type"] == "post") & (df["geneID"] == geneID)]["var"].to_numpy()
df_pred = df.loc[(df["modelID"] == modelID) & (df["type"] == "pred") & (df["geneID"] == geneID)]["var"].to_numpy()
weights_post = np.ones_like(df_post) / df_post.shape[0]
weights_pred = np.ones_like(df_pred) / df_pred.shape[0]
v_ratio = str(round(np.sum(df_post > df_pred) / df_post.size * 100, 2))
bins = np.histogram(
                    np.hstack([df_post,df_pred]), bins=100
                )[1]

ax1.set_title("A", loc="left",fontsize=26)
ax1.set_xlabel("variance in number of substitutions",fontsize=14)
ax1.tick_params(axis='x', labelsize=14)  # Replace 12 with the desired font size

ax1.hist(
    [df_post],
    bins=bins,
    color="#1f77b4",
    alpha=0.5,
    stacked=False,
    weights=[weights_post],
    label=["% " + v_ratio],
)
ax1.hist(
    [df_pred],
    bins=bins,
    color="#ff7f0e",
    alpha=0.5,
    stacked=False,
    weights=[weights_post],
)
_ = ax1.set_yticks([])
_ = ax1.set_yticklabels([])
ax1.set_yticks([])  # Remove the y-ticks
ax1.set_ylabel('')  # Optionally remove the y-axis label
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_visible(False)
ax1.spines['bottom'].set_visible(True)
# ax1.axvline(x=5.09, color='black', linestyle='--', label='parsimony') 
##############
# B
##############

df_ = pd.read_csv(f"{ROOT_dir}/outputs/empirical/sub_mat/stats/AA/{geneID}-{modelID}-A_suffstatmap.tsv", sep="\t", index_col=0).set_index(keys=["geneID","modelID","repID","type"])
assert df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].sum(axis=1).sum() == pytest.approx(df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].shape[0])
assert df_.loc[(geneID,modelID,"A","pred"),list(get_set_of_keys())].sum(axis=1).sum() == pytest.approx(df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].shape[0])
dist: np.array = ((df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].to_numpy() - df_.loc[(geneID,modelID,"A","pred"),list(get_set_of_keys())].to_numpy())**2).sum(axis=1)
print("rho",modelID,geneID,np.mean(dist),np.std(dist))
dist: np.array = ((df_.loc[(geneID,modelID,"A","post"),AA].to_numpy() - df_.loc[(geneID,modelID,"A","pred"),AA].to_numpy())**2).sum(axis=1)
print("phi",modelID,geneID,np.mean(dist),np.std(dist)) 

df_rho = df_[list(get_set_of_keys())+["mcmcID"]].groupby(by=["geneID","modelID","repID","type"]).agg([np.sum]).droplevel(level=1,axis=1)      
df_rho["row_sum"] = df_rho[list(get_set_of_keys())].sum(axis=1)
df_rho.loc[:, list(get_set_of_keys())] =  df_rho[list(get_set_of_keys())].apply(lambda x: x / df_rho["row_sum"])[list(get_set_of_keys())]
df_rho = df_rho.drop(labels=["row_sum"],axis=1)
for x in range(1,21,1):
    try:
        a = AA[x-1]
    except Exception as e:
        print(a)
        raise RuntimeError
    for y in range(21-x,0,-1):
        try:
            b = AA[::-1][y-1]
        except Exception as e: 
            print(y)
            raise RuntimeError
        if a == b:
            pass
        else:
            s = a+"<>"+b if a < b else b+"<>"+a
            _ = ax2.scatter(x=x,y=y,s=df_rho.loc[(geneID, modelID, "A","post"),s]*2000,marker='o',color="#1f77b4")

for y in range(20,1,-1):
    try:
        b = AA[::-1][y-1]
    except Exception as e:
        print(y)
        raise RuntimeError
    for x in range(21-y,21,1):
        try:
            a = AA[x-1]
        except Exception as e:
            print(a)
            raise RuntimeError
        if a == b:
            pass
        else:
            s = a+"<>"+b if a < b else b+"<>"+a
            _ = ax2.scatter(x=x,y=y,s=df_rho.loc[(geneID, modelID, "A","pred"),s]*2000,marker='o',color="#ff7f0e")
_ = ax2.set_xticks([i+1 for i in range(0,len(AA))])
_ = ax2.set_xticklabels(AA)
_ = ax2.set_yticks([i+1 for i in range(0,len(AA))])
_ = ax2.set_yticklabels(reversed(AA))
_ = ax2.tick_params(axis='y', labelsize=14)
_ = ax2.tick_params(axis='x', labelsize=14)
_ = ax2.set_title("B", loc="left",fontsize=26)
ax2.set_aspect('equal', adjustable='box')
ax2.set_xlim((0,21))
ax2.set_ylim((0,21))
ax2.set_xlabel("", )#fontweight ='bold', fontsize = 15


##############
# C
##############

df_phi = df_[AA+["mcmcID"]].groupby(by=["geneID","modelID","repID","type"]).agg([np.sum]).droplevel(level=1,axis=1)      
df_phi["row_sum"] = df_phi[AA].sum(axis=1)
df_phi.loc[:, AA] =  df_phi[AA].apply(lambda x: x / df_phi["row_sum"])[AA]
df_phi = df_phi.drop(labels=["row_sum"],axis=1)

barWidth = 0.25
df_phi_pred_h = df_phi.loc[(geneID, modelID, "A","pred"),AA].to_list()
df_phi_post_h = df_phi.loc[(geneID, modelID, "A","post"),AA].to_list()
#df_phi_pred_err = df_phi.loc[(df_phi["type"]=="pred")].iloc[:,4:].std().to_list()
#df_phi_post_err = df_phi.loc[(df_phi["type"]=="post")].iloc[:,4:].std().to_list()
df_phi_post_p = np.arange(len(df_phi_post_h))
df_phi_pred_p = [x + barWidth for x in df_phi_post_p]

ax3.bar(df_phi_post_p, df_phi_post_h, color ='#1f77b4', width = barWidth,
        label ='augmented',align='center', alpha=1, ecolor='grey',error_kw=dict(lw=barWidth/10, capsize=barWidth, capthick=barWidth/10))#yerr=df_phi_post_err
ax3.bar(df_phi_pred_p, df_phi_pred_h, color ='#ff7f0e', width = barWidth,
        label ='predictive',align='center', alpha=1, ecolor='gray',error_kw=dict(lw=barWidth/10, capsize=barWidth, capthick=barWidth/10)
)

ax3.set_xlabel('', )#fontweight ='bold', fontsize = 15
if k == 4:
    ax3.set_ylabel('posterior means',)# fontweight ='bold', fontsize = 15
_ = ax3.tick_params(axis='x', labelsize=14)
_ = ax3.set_xticks([r + barWidth for r in range(len(df_phi_post_h))], AA)
_ = ax3.set_title("C", loc="left",fontsize=26)      
ax3.set_yticks([])  # Remove the y-ticks
ax3.set_ylabel('')  # Optionally remove the y-axis label
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['left'].set_visible(False)
ax3.spines['bottom'].set_visible(True)

##############
# D
##############
logo = get_logo(list_of_profiles_post,colours=expand_colors(base_colors,AA_COLORS_SLL),layout=get_base_logo_layout(1,12,12),ylim=1)
logo.plotly_figure.write_image("plotly_figure1D.png")
import matplotlib.image as mpimg
image = mpimg.imread("plotly_figure1D.png")
ax4.imshow(image)
_ = ax4.set_title("D", loc="left",fontsize=26)    
ax4.set_xticks([])  # Remove x-ticks
ax4.set_yticks([]) 
ax4.axis('off')

##############
# E
##############
logo = get_logo(list_of_profiles_pred,colours=expand_colors(base_colors,AA_COLORS_SLL),layout=get_base_logo_layout(1,12,12),ylim=1)
logo.plotly_figure.write_image("plotly_figure1E.png")
import matplotlib.image as mpimg
image = mpimg.imread("plotly_figure1E.png")
ax5.imshow(image)
_ = ax5.set_title("E", loc="left",fontsize=26) 
ax5.set_xticks([])  # Remove x-ticks
ax5.set_yticks([])
ax5.axis('off')

# Adjust layout to prevent overlap
plt.tight_layout()
##plt.savefig(f"{ROOT_dir}/reports/figure1.pdf",dpi=360)

plt.show()

In [ ]:
list_of_rho = df_LG_rho.index.to_list()
X_JCAA_rho = df_rho.loc[(geneID, modelID, "A","post")][list_of_rho].to_numpy().reshape(-1,)
y = df_LG_rho.loc[list_of_rho,:].to_numpy().reshape(-1,)
res = stats.pearsonr(X_JCAA_rho, y)
print(res)


In [ ]:
X_JCAA_phi = df_phi.loc[(geneID, modelID, "A","post")][AA].to_numpy().reshape(-1,)
y = np.array(LG_Stat[0]).reshape(-1,)
res = stats.pearsonr(X_JCAA_phi, y)
print(res)

In [ ]:
MODELID = ["CATC10GTR4GAA"]
GENEID = ["BGLOBIN"]


list_of_df = []
for modelID in MODELID:
    for geneID in GENEID: 
        for repID in ["A",]:
            df: pd.DataFrame = pd.read_csv(f"{ROOT_dir}/outputs/empirical/site_profile/stats/AA/{geneID}-{modelID}-{repID}_suffstatmap.tsv",sep="\t")
            # df_post = df.loc[df["type"] == "post"]
            # df_pred = df.loc[df["type"] == "pred"]
            list_of_df += [df]
df_concat  = pd.concat(list_of_df)       

list_of_df_centroids = []
dict_of_stats = {}
l = 0
for modelID in MODELID:
    for typeID in ["post", "pred"]:
        for geneID in GENEID:
            for repID in ["A",]:
                df_cur = df_concat.loc[(df_concat["type"]==typeID)&(df_concat["modelID"]==modelID)&(df_concat["geneID"]==geneID)&(df_concat["repID"]==repID)][amino_acids_upper[:-1]]
                scaler = StandardScaler()
                df_cur_scaled = scaler.fit_transform(df_cur)
                
                for k in [10,60]:
                    kmeans = KMeans(n_clusters=k, random_state=42)

                    # Fit the model
                    kmeans.fit(df_cur_scaled)
                    centroids = kmeans.cluster_centers_
                    cluster_labels = kmeans.labels_

                    dict_of_count = {}

                    for i in range(k):
                        dict_of_count[i] = np.sum(cluster_labels == i) / len(cluster_labels)

                    centroids_inverse_transformed = scaler.inverse_transform(centroids)
                    df_centroids_inverse_transformed = pd.DataFrame(data=centroids_inverse_transformed, columns=amino_acids_upper[:-1])
                    df_centroids_inverse_transformed["typeID"] = [typeID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["modelID"] = [modelID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["geneID"] = [geneID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["repID"] = [repID] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["k"] = [k] * df_centroids_inverse_transformed.shape[0]
                    df_centroids_inverse_transformed["weight"] = list(dict_of_count.values())

                    entropy_mappings = compute_entropy(df_centroids_inverse_transformed[amino_acids_upper[:-1]])

                    list_of_df_centroids += [df_centroids_inverse_transformed[["typeID","geneID", "modelID", "repID","k","weight"] + amino_acids_upper[:-1]]]
                    # if k == 10:
                    #     scaled_C10_profiles = scaler.transform(df_C10_profiles)
                    #     cluster_labels = kmeans.predict(scaled_C10_profiles)
                    #     cluster_labels_set = set(cluster_labels)
                    #     entropy_mixture = compute_entropy(df_C10_profiles)
                    
                    # elif k == 60:
                    #     scaled_C60_profiles = scaler.transform(df_C60_profiles)
                    #     cluster_labels = kmeans.predict(scaled_C60_profiles)
                    #     cluster_labels_set = set(cluster_labels)
                    #     entropy_mixture = compute_entropy(df_C60_profiles)
                    
                    dict_of_stats[l] = {
                        "typeID": typeID, # "post", "pred
                        "modelID": modelID,
                        "geneID": geneID,
                        "repID": repID,
                        "k": "Nclusters"+str(k),
                        "cluster_labels": cluster_labels,
                        "cluster_weights": list(dict_of_count.values()),
                        "entropy_mappings_mean": entropy_mappings.mean(),
                        

                    }
                    l+=1
df_centroids = pd.concat(list_of_df_centroids)

list_of_profiles_post = []
list_of_profiles_pred = []
for row in df_centroids.iterrows():
    typeID = row[1][0]
    profileID =row[1][4]
    if profileID == 10 and typeID == "post":
        dict_of_profile = {}
        sum_ = 0
        for i in range(6,26):
            aaID = row[1].index[i]
            aaPref = row[1][i]
            # print(aaPref)
            dict_of_profile.update({aaID : aaPref})
            sum_ += aaPref
        assert abs(sum_ - 1) < 0.01, f"Sum is {sum_}, but it should be 1."
        list_of_profiles_post += [dict_of_profile]

    if profileID == 10 and typeID == "pred":
        dict_of_profile = {}
        sum_ = 0
        for i in range(6,26):
            aaID = row[1].index[i]
            aaPref = row[1][i]
            # print(aaPref)
            dict_of_profile.update({aaID : aaPref})
            sum_ += aaPref
        assert abs(sum_ - 1) < 0.01, f"Sum is {sum_}, but it should be 1."
        list_of_profiles_pred += [dict_of_profile]

In [ ]:
# Create a figure
fig = plt.figure(figsize=(15, 9))

# Define a GridSpec layout (3 rows, 2 columns)
gs = gridspec.GridSpec(8, 2, width_ratios=[1, 1])

# Left section: 3 subplots stacked vertically in the first column
ax1 = plt.subplot(gs[0:2, 0])  # First subplot in the first column
ax2 = plt.subplot(gs[2:6, 0])  # Second subplot in the first column
ax3 = plt.subplot(gs[6:8, 0])  # Third subplot in the first column

# Right section: 2 subplots stacked vertically in the second column
ax4 = plt.subplot(gs[0:4, 1])  # First and second row, merged
ax5 = plt.subplot(gs[4:8, 1])    # Third row, second column

# Set titles or plot some data for clarity
# ax1.set_title('Left Plot 1')
# ax2.set_title('Left Plot 2')
# ax3.set_title('Left Plot 3')
# ax4.set_title('Right Plot 1 (Merged Rows)')
# ax5.set_title('Right Plot 2')

##############
# A
##############

df: pd.DataFrame = pd.read_csv(f"{ROOT_dir}/outputs/empirical/site_rates/stats/AA/{geneID}-{modelID}-A_suffstatmap.tsv",sep="\t")
df_post = df.loc[(df["modelID"] == modelID) & (df["type"] == "post") & (df["geneID"] == geneID)]["var"].to_numpy()
df_pred = df.loc[(df["modelID"] == modelID) & (df["type"] == "pred") & (df["geneID"] == geneID)]["var"].to_numpy()
weights_post = np.ones_like(df_post) / df_post.shape[0]
weights_pred = np.ones_like(df_pred) / df_pred.shape[0]
v_ratio = str(round(np.sum(df_post > df_pred) / df_post.size * 100, 2))
bins = np.histogram(
                    np.hstack([df_post,df_pred]), bins=100
                )[1]

ax1.set_title("A", loc="left",fontsize=26)
ax1.set_xlabel("variance in number of substitutions",fontsize=14)
ax1.tick_params(axis='x', labelsize=14)  # Replace 12 with the desired font size

ax1.hist(
    [df_post],
    bins=bins,
    color="#1f77b4",
    alpha=0.5,
    stacked=False,
    weights=[weights_post],
    label=["% " + v_ratio],
)
ax1.hist(
    [df_pred],
    bins=bins,
    color="#ff7f0e",
    alpha=0.5,
    stacked=False,
    weights=[weights_post],
)
_ = ax1.set_yticks([])
_ = ax1.set_yticklabels([])
ax1.set_yticks([])  # Remove the y-ticks
ax1.set_ylabel('')  # Optionally remove the y-axis label
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_visible(False)
ax1.spines['bottom'].set_visible(True)
# ax1.axvline(x=5.09, color='black', linestyle='--', label='parsimony') 
##############
# B
##############

df_ = pd.read_csv(f"{ROOT_dir}/outputs/empirical/sub_mat/stats/AA/{geneID}-{modelID}-A_suffstatmap.tsv", sep="\t", index_col=0).set_index(keys=["geneID","modelID","repID","type"])
assert df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].sum(axis=1).sum() == pytest.approx(df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].shape[0])
assert df_.loc[(geneID,modelID,"A","pred"),list(get_set_of_keys())].sum(axis=1).sum() == pytest.approx(df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].shape[0])
dist: np.array = ((df_.loc[(geneID,modelID,"A","post"),list(get_set_of_keys())].to_numpy() - df_.loc[(geneID,modelID,"A","pred"),list(get_set_of_keys())].to_numpy())**2).sum(axis=1)
print("rho",modelID,geneID,np.mean(dist),np.std(dist))
dist: np.array = ((df_.loc[(geneID,modelID,"A","post"),AA].to_numpy() - df_.loc[(geneID,modelID,"A","pred"),AA].to_numpy())**2).sum(axis=1)
print("phi",modelID,geneID,np.mean(dist),np.std(dist)) 

df_rho = df_[list(get_set_of_keys())+["mcmcID"]].groupby(by=["geneID","modelID","repID","type"]).agg([np.sum]).droplevel(level=1,axis=1)      
df_rho["row_sum"] = df_rho[list(get_set_of_keys())].sum(axis=1)
df_rho.loc[:, list(get_set_of_keys())] =  df_rho[list(get_set_of_keys())].apply(lambda x: x / df_rho["row_sum"])[list(get_set_of_keys())]
df_rho = df_rho.drop(labels=["row_sum"],axis=1)
for x in range(1,21,1):
    try:
        a = AA[x-1]
    except Exception as e:
        print(a)
        raise RuntimeError
    for y in range(21-x,0,-1):
        try:
            b = AA[::-1][y-1]
        except Exception as e: 
            print(y)
            raise RuntimeError
        if a == b:
            pass
        else:
            s = a+"<>"+b if a < b else b+"<>"+a
            _ = ax2.scatter(x=x,y=y,s=df_rho.loc[(geneID, modelID, "A","post"),s]*2000,marker='o',color="#1f77b4")

for y in range(20,1,-1):
    try:
        b = AA[::-1][y-1]
    except Exception as e:
        print(y)
        raise RuntimeError
    for x in range(21-y,21,1):
        try:
            a = AA[x-1]
        except Exception as e:
            print(a)
            raise RuntimeError
        if a == b:
            pass
        else:
            s = a+"<>"+b if a < b else b+"<>"+a
            _ = ax2.scatter(x=x,y=y,s=df_rho.loc[(geneID, modelID, "A","pred"),s]*2000,marker='o',color="#ff7f0e")
_ = ax2.set_xticks([i+1 for i in range(0,len(AA))])
_ = ax2.set_xticklabels(AA)
_ = ax2.set_yticks([i+1 for i in range(0,len(AA))])
_ = ax2.set_yticklabels(reversed(AA))
_ = ax2.tick_params(axis='y', labelsize=14)
_ = ax2.tick_params(axis='x', labelsize=14)
_ = ax2.set_title("B", loc="left",fontsize=26)
ax2.set_aspect('equal', adjustable='box')
ax2.set_xlim((0,21))
ax2.set_ylim((0,21))
ax2.set_xlabel("", )#fontweight ='bold', fontsize = 15


##############
# C
##############

df_phi = df_[AA+["mcmcID"]].groupby(by=["geneID","modelID","repID","type"]).agg([np.sum]).droplevel(level=1,axis=1)      
df_phi["row_sum"] = df_phi[AA].sum(axis=1)
df_phi.loc[:, AA] =  df_phi[AA].apply(lambda x: x / df_phi["row_sum"])[AA]
df_phi = df_phi.drop(labels=["row_sum"],axis=1)

barWidth = 0.25
df_phi_pred_h = df_phi.loc[(geneID, modelID, "A","pred"),AA].to_list()
df_phi_post_h = df_phi.loc[(geneID, modelID, "A","post"),AA].to_list()
#df_phi_pred_err = df_phi.loc[(df_phi["type"]=="pred")].iloc[:,4:].std().to_list()
#df_phi_post_err = df_phi.loc[(df_phi["type"]=="post")].iloc[:,4:].std().to_list()
df_phi_post_p = np.arange(len(df_phi_post_h))
df_phi_pred_p = [x + barWidth for x in df_phi_post_p]

ax3.bar(df_phi_post_p, df_phi_post_h, color ='#1f77b4', width = barWidth,
        label ='augmented',align='center', alpha=1, ecolor='grey',error_kw=dict(lw=barWidth/10, capsize=barWidth, capthick=barWidth/10))#yerr=df_phi_post_err
ax3.bar(df_phi_pred_p, df_phi_pred_h, color ='#ff7f0e', width = barWidth,
        label ='predictive',align='center', alpha=1, ecolor='gray',error_kw=dict(lw=barWidth/10, capsize=barWidth, capthick=barWidth/10)
)

ax3.set_xlabel('', )#fontweight ='bold', fontsize = 15
if k == 4:
    ax3.set_ylabel('posterior means',)# fontweight ='bold', fontsize = 15
_ = ax3.tick_params(axis='x', labelsize=14)
_ = ax3.set_xticks([r + barWidth for r in range(len(df_phi_post_h))], AA)
_ = ax3.set_title("C", loc="left",fontsize=26)      
ax3.set_yticks([])  # Remove the y-ticks
ax3.set_ylabel('')  # Optionally remove the y-axis label
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['left'].set_visible(False)
ax3.spines['bottom'].set_visible(True)

##############
# D
##############
logo = get_logo(list_of_profiles_post,colours=expand_colors(base_colors,AA_COLORS_SLL),layout=get_base_logo_layout(1,12,12),ylim=1)
logo.plotly_figure.write_image("plotly_figure2D.png")
import matplotlib.image as mpimg
image = mpimg.imread("plotly_figure2D.png")
ax4.imshow(image)
_ = ax4.set_title("D", loc="left",fontsize=26)    
ax4.set_xticks([])  # Remove x-ticks
ax4.set_yticks([]) 
ax4.axis('off')

##############
# E
##############
logo = get_logo(list_of_profiles_pred,colours=expand_colors(base_colors,AA_COLORS_SLL),layout=get_base_logo_layout(1,12,12),ylim=1)
logo.plotly_figure.write_image("plotly_figure2E.png")
import matplotlib.image as mpimg
image = mpimg.imread("plotly_figure2E.png")
ax5.imshow(image)
_ = ax5.set_title("E", loc="left",fontsize=26) 
ax5.set_xticks([])  # Remove x-ticks
ax5.set_yticks([])
ax5.axis('off')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.savefig(f"{ROOT_dir}/reports/figure2.pdf",dpi=360)
plt.show()

In [ ]:
X_CATGTR4GAA_rho_post = df_rho.loc[(geneID, modelID, "A","post")][list_of_rho].to_numpy().reshape(-1,)
y_CATGTR4GAA_rho_pred = df_rho.loc[(geneID, modelID, "A","pred")][list_of_rho].to_numpy().reshape(-1,)
res = stats.pearsonr(X_CATGTR4GAA_rho_post, y_CATGTR4GAA_rho_pred)
print(res)

In [ ]:
X_CATGTR4GAA_phi_post = df_phi.loc[(geneID, modelID, "A","post")][AA].to_numpy().reshape(-1,)
y_CATGTR4GAA_phi_pred = df_phi.loc[(geneID, modelID, "A","pred")][AA].to_numpy().reshape(-1,)
res = stats.pearsonr(X_CATGTR4GAA_phi_post, y_CATGTR4GAA_phi_pred)
print(res)

In [ ]:
list_of_rho = df_LG_rho.index.to_list()
X_CATGTR4GAA_rho = df_rho.loc[(geneID, modelID, "A","post")][list_of_rho].to_numpy().reshape(-1,)
y = df_LG_rho.loc[list_of_rho,:].to_numpy().reshape(-1,)
res = stats.pearsonr(X_CATGTR4GAA_rho, y)
print(res)

In [ ]:
X_CATGTR4GAA_phi = df_phi.loc[(geneID, modelID, "A","post")][AA].to_numpy().reshape(-1,)
y = np.array(LG_Stat[0]).reshape(-1,)
res = stats.pearsonr(X_CATGTR4GAA_phi, y)
print(res)

In [ ]:
res = stats.pearsonr(X_JCAA_rho, X_CATGTR4GAA_rho)
print(res)

In [ ]:
res = stats.pearsonr(X_JCAA_phi, X_CATGTR4GAA_phi)
print(res)